# Week 4 - Baseline Model

Train and evaluate one Linear Regression model using the discussed features from the fixed-cleaned Dataset A files.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the fixed-cleaned training, validation, and test files created by notebook 02.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "data" / "processed"

paths = {
    "Training": DATA_DIR / "train_clean.csv",
    "Validation": DATA_DIR / "validation_clean.csv",
    "Test": DATA_DIR / "test_clean.csv",
}
missing_files = [str(path) for path in paths.values() if not path.exists()]
assert not missing_files, f"Run notebook 02 first. Missing files: {missing_files}"
splits = {name: pd.read_csv(path, low_memory=False) for name, path in paths.items()}

split_summary = pd.DataFrame([
    [name, data["SourceMonth"].min(), data["SourceMonth"].max(), len(data)]
    for name, data in splits.items()
], columns=["Split", "First month", "Last month", "Rows"]).set_index("Split")
display(split_summary)

In [ ]:
# Select only the features whose rules are marked Discussed.
TARGET = "ClosePrice" # what model predicts
BINARY_FEATURES = [
    "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "PoolPrivateYN", "ViewYN",
]
MODEL_FEATURES = ["LivingArea"] + BINARY_FEATURES # continuous (living area) + binary type (YN fields)
REQUIRED_COLUMNS = MODEL_FEATURES + [TARGET, "SourceMonth"]

# missing value effect
for name, data in splits.items():
    missing = sorted(set(REQUIRED_COLUMNS) - set(data.columns))
    assert not missing, f"{name} is missing columns: {missing}"
    data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
    data[MODEL_FEATURES] = data[MODEL_FEATURES].apply(pd.to_numeric, errors="coerce")

feature_plan = pd.DataFrame([
    ["LivingArea", "Median imputation, then standard scaling"],
    [", ".join(BINARY_FEATURES), "Training-mode imputation plus one missing flag per field"],
], columns=["Source feature", "Training-fitted pipeline rule"])
display(feature_plan)

In [ ]:
# Build preprocessing that learns every imputation and scaling value from training data.
living_area_pipeline = Pipeline([
    ("median_imputer", SimpleImputer(strategy="median")), #used median, less affected by unusually large or small living area
    ("scaler", StandardScaler()), # (LivingArea - training mean) / training standard deviation
])

def binary_pipeline():
    return FeatureUnion([
        ("mode_imputed_value", SimpleImputer(strategy="most_frequent")), # mode is the most common valid category in the training data
        ("missing_indicator", MissingIndicator(features="all")),
    ])

preprocessor = ColumnTransformer([
    ("living_area", living_area_pipeline, ["LivingArea"]),
    *[(f"binary_{column}", binary_pipeline(), [column]) for column in BINARY_FEATURES],
])
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("linear_regression", LinearRegression()),
]) # guarantees the model receives transformed features instead of the original missing values.

In [ ]:
# Fit the complete preprocessing and Linear Regression pipeline on training data only.
train = splits["Training"]
validation = splits["Validation"]
test = splits["Test"]

X_train, y_train = train[MODEL_FEATURES], train[TARGET]
X_test, y_test = test[MODEL_FEATURES], test[TARGET]

assert y_train.notna().all() and y_test.notna().all()
baseline_model.fit(X_train, y_train)
print("Linear Regression fitted. Validation remains reserved for later model or window comparisons.")

In [ ]:
# Evaluate the held-out test month with R-squared and record the baseline result.
test_predictions = baseline_model.predict(X_test)
test_r2 = r2_score(y_test, test_predictions)
model_ready_features = ["LivingArea_scaled"]
for column in BINARY_FEATURES:
    model_ready_features.extend([f"{column}_imputed", f"{column}_missing"])

baseline_results = pd.DataFrame([{
    "Model": "Linear Regression",
    "Feature policy": "Discussed fields only",
    "Training months": f"{train['SourceMonth'].min()} to {train['SourceMonth'].max()}",
    "Test month": test["SourceMonth"].max(),
    "Training rows": len(train),
    "Test rows": len(test),
    "Source features": len(MODEL_FEATURES), # LivingArea + 5 binary
    "Model-ready features": len(model_ready_features), # living area + 5 binary + 5 binary missing indicators
    "Test R-squared": test_r2,
}])
display(baseline_results.style.format({"Test R-squared": "{:.4f}"}))
print(f"Test R-squared: {test_r2:.4f}")

In [ ]:
# Verify the cleaned inputs and the expected 11 model-ready feature columns.
for name, data in splits.items():
    assert data[TARGET].notna().all() and data[TARGET].gt(0).all()
    assert data["LivingArea"].dropna().gt(0).all()
    for column in BINARY_FEATURES:
        assert set(data[column].dropna().unique()).issubset({0.0, 1.0})

X_train_ready = baseline_model.named_steps["preprocessor"].transform(X_train)
assert X_train_ready.shape == (len(train), len(model_ready_features))
print("All baseline checks passed.")
print("Model-ready features:", model_ready_features)